# Final Test Set Comparison

**Project:** CaféScan — Coffee Leaf Disease Detection  
**Purpose:** Load and display the final test set results, comparison figures, and overfitting analysis
across all 4 architectures.

**Test Set Results (Macro-F1):**
| Model | Macro-F1 |
|---|---|
| ViT | **99.65%** |
| EfficientNet-B0 | 95.40% |
| MobileNet | 91.38% |
| ResNet-50 | 80.39% |

**Classes:** healthy, rust, cercospora, miner, phoma

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from pathlib import Path
from IPython.display import display, Image as IPyImage

plt.style.use('seaborn-v0_8-whitegrid')

ROOT = Path(".")
RESULTS = ROOT / "results"
TABLES = RESULTS / "tables"
FIGURES = RESULTS / "figures"

MODELS = ["efficientnet_b0", "resnet50", "vit", "mobilenet"]
MODEL_LABELS = {
    "efficientnet_b0": "EfficientNet-B0",
    "resnet50": "ResNet-50",
    "vit": "ViT",
    "mobilenet": "MobileNet",
}
CLASSES = ["healthy", "rust", "cercospora", "miner", "phoma"]

cmap = plt.get_cmap("tab10")
MODEL_COLORS = {m: cmap(i) for i, m in enumerate(MODELS)}

print("Imports OK. ROOT:", ROOT.resolve())

## 1. Test Summary — All Metrics

Load `results/tables/test_summary.csv` and display with per-column best-value highlighting.

In [ ]:
test_df = pd.read_csv(TABLES / "test_summary.csv")
test_df = test_df.set_index("model")
test_df.index = test_df.index.map(lambda m: MODEL_LABELS.get(m, m))

# Rename columns for display
col_rename = {
    "accuracy": "Accuracy",
    "macro_f1": "Macro-F1",
    "weighted_f1": "Weighted-F1",
    "macro_precision": "Macro-Precision",
    "macro_recall": "Macro-Recall",
    "f1_healthy": "F1-Healthy",
    "f1_rust": "F1-Rust",
    "f1_cercospora": "F1-Cercospora",
    "f1_miner": "F1-Miner",
    "f1_phoma": "F1-Phoma",
}
test_df = test_df.rename(columns=col_rename)

# Highlight best value (max) per column in green
def highlight_best(col):
    is_best = col == col.max()
    return ["background-color: #c6efce; font-weight: bold" if v else "" for v in is_best]

styled_test = (
    test_df.style
    .apply(highlight_best, axis=0)
    .format("{:.4f}")
    .set_caption("Test Set Metrics — All Models (best per column highlighted in green)")
    .set_table_styles([{
        "selector": "caption",
        "props": [("font-size", "14px"), ("font-weight", "bold"), ("text-align", "left")]
    }])
)
display(styled_test)

## 2. Test Comparison Figure

Pre-generated bar chart comparing overall test metrics across all models.

In [ ]:
fig_path = FIGURES / "test_comparison.png"
img = plt.imread(str(fig_path))
fig, ax = plt.subplots(figsize=(12, 6))
ax.imshow(img)
ax.axis("off")
ax.set_title("Test Set Comparison — Overall Metrics", fontsize=14, fontweight="bold", pad=10)
plt.tight_layout()
plt.show()
plt.close()

## 3. Per-Class F1 Comparison Figure

In [ ]:
fig_path2 = FIGURES / "test_per_class_f1.png"
img2 = plt.imread(str(fig_path2))
fig2, ax2 = plt.subplots(figsize=(12, 6))
ax2.imshow(img2)
ax2.axis("off")
ax2.set_title("Per-Class F1 Scores — All Models", fontsize=14, fontweight="bold", pad=10)
plt.tight_layout()
plt.show()
plt.close()

## 4. Overfitting Analysis — Validation vs Test F1

Load `results/tables/val_vs_test.csv` and color rows by status:  
- **OK** — gap is within acceptable range (light green)  
- **OVERFIT?** — gap exceeds threshold (light red)

In [ ]:
gap_df = pd.read_csv(TABLES / "val_vs_test.csv")
gap_df["model"] = gap_df["model"].map(lambda m: MODEL_LABELS.get(m, m))
gap_df = gap_df.rename(columns={
    "model": "Model",
    "val_f1": "Val Macro-F1",
    "test_f1": "Test Macro-F1",
    "gap": "Gap (Val - Test)",
    "status": "Status",
})
gap_df = gap_df.set_index("Model")

def highlight_status(row):
    status = row["Status"]
    if status == "OK":
        color = "background-color: #c6efce"  # light green
    elif "OVERFIT" in str(status).upper():
        color = "background-color: #ffc7ce"  # light red
    else:
        color = ""
    return [color] * len(row)

numeric_cols = [c for c in gap_df.columns if c != "Status"]

styled_gap = (
    gap_df.style
    .apply(highlight_status, axis=1)
    .format({c: "{:.4f}" for c in numeric_cols})
    .set_caption("Validation vs Test Macro-F1 — Overfitting Check")
    .set_table_styles([{
        "selector": "caption",
        "props": [("font-size", "14px"), ("font-weight", "bold"), ("text-align", "left")]
    }])
)
display(styled_gap)

## 5. Per-Class F1 Heatmap

Heatmap where **rows = models**, **columns = classes**, **values = F1 scores**.
Extracted from the `f1_*` columns of `test_summary.csv`.

In [ ]:
# Reload raw test_df for heatmap
raw_df = pd.read_csv(TABLES / "test_summary.csv").set_index("model")
f1_cols = [f"f1_{cls}" for cls in CLASSES]
heatmap_data = raw_df[f1_cols].copy()
heatmap_data.index = [MODEL_LABELS.get(m, m) for m in heatmap_data.index]
heatmap_data.columns = [c.replace("f1_", "").title() for c in heatmap_data.columns]

fig, ax = plt.subplots(figsize=(9, 4))
im = ax.imshow(heatmap_data.values, cmap="RdYlGn", vmin=0.6, vmax=1.0, aspect="auto")

# Axis labels
ax.set_xticks(range(len(CLASSES)))
ax.set_xticklabels(heatmap_data.columns, fontsize=11)
ax.set_yticks(range(len(heatmap_data)))
ax.set_yticklabels(heatmap_data.index, fontsize=11)

# Annotate cells
for i in range(len(heatmap_data)):
    for j in range(len(CLASSES)):
        val = heatmap_data.values[i, j]
        text_color = "black" if 0.65 < val < 0.93 else "white"
        ax.text(j, i, f"{val:.3f}", ha="center", va="center",
                fontsize=10, color=text_color, fontweight="bold")

cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("F1 Score", fontsize=10)

ax.set_title("Per-Class F1 Score Heatmap — Test Set", fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("Disease Class", fontsize=11)
ax.set_ylabel("Model", fontsize=11)

plt.tight_layout()
plt.savefig(FIGURES / "nb_per_class_f1_heatmap.png", dpi=120, bbox_inches="tight")
plt.show()
plt.close()

## 6. Overall Performance Bar Chart (Custom)

In [ ]:
# Bar chart: Macro-F1 and Accuracy for each model on test set
raw_df2 = pd.read_csv(TABLES / "test_summary.csv")
raw_df2["label"] = raw_df2["model"].map(lambda m: MODEL_LABELS.get(m, m))
raw_df2 = raw_df2.sort_values("macro_f1", ascending=False)

x = np.arange(len(raw_df2))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, raw_df2["macro_f1"], width,
               label="Macro-F1", color="steelblue", alpha=0.88)
bars2 = ax.bar(x + width/2, raw_df2["accuracy"], width,
               label="Accuracy", color="darkorange", alpha=0.88)

# Value labels on bars
for bar in bars1:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.003, f"{h:.4f}",
            ha="center", va="bottom", fontsize=9, fontweight="bold")
for bar in bars2:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.003, f"{h:.4f}",
            ha="center", va="bottom", fontsize=9, fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels(raw_df2["label"], fontsize=11)
ax.set_ylabel("Score", fontsize=12)
ax.set_title("Test Set Performance — Macro-F1 vs Accuracy (sorted)", fontsize=13, fontweight="bold")
ax.set_ylim(0.6, 1.05)
ax.legend(fontsize=11)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.2f}"))

plt.tight_layout()
plt.savefig(FIGURES / "nb_test_bar_summary.png", dpi=120, bbox_inches="tight")
plt.show()
plt.close()

## 7. Conclusions

### Model Ranking (Test Macro-F1)
1. **ViT — 99.65%** — Best by a wide margin. Global self-attention captures long-range
   leaf texture patterns that CNNs may miss. Near-perfect detection across all 5 disease classes.
2. **EfficientNet-B0 — 95.40%** — Strong performance with far fewer parameters than ViT.
   Compound scaling delivers excellent accuracy-efficiency tradeoff. Production-viable.
3. **MobileNet — 91.38%** — Acceptable for edge/mobile deployment. Lightweight but leaves
   approximately 4 percentage points of F1 on the table vs EfficientNet.
4. **ResNet-50 — 80.39%** — Underperformed relative to expectations. The plain residual
   architecture without compound scaling or attention struggles with fine-grained disease textures.

### Overfitting
The `val_vs_test.csv` gap analysis reveals whether any model over-tuned to the validation set.
A positive gap (Val > Test) is expected and normal; a large gap suggests overfitting.
All models trained with early stopping on validation Macro-F1 should show minimal overfitting.

### Per-Class Difficulty
From the heatmap, the **miner** (leaf miner) class tends to be the hardest across all architectures
due to its subtle, localized damage pattern. **Rust** is typically the easiest to detect given
its distinctive orange pustule appearance.

### Recommendation
- **Research/accuracy-critical use:** ViT  
- **Production/cloud deployment:** EfficientNet-B0  
- **Edge/mobile deployment:** MobileNet  
- **Avoid:** ResNet-50 for this task without further architecture modifications